# tp-12 予測オッズでの馬券最適化（実運用に最も近い設定）

**考え方**: 実運用では、購入を決める時点で確定オッズは分からない。tp-08/09 は**確定オッズで買い目を選んで確定オッズで精算**しており、締切直前の動きを無視した楽観的な設定だった。
ここでは、判断に使うオッズを**予測オッズ**（そのレースのオッズ関連情報を一切使わずに予測した値）にして、精算だけを実際の払戻（確定）で行う。

## ルール（情報の漏れを防ぐ）
- **オッズ予測の特徴量にオッズ・人気は含めない**（列名を検査して含まれていれば停止）。履歴・血統・コース・調教など、レース前に分かる情報だけ。
- オッズ予測は**ローリング**: 検証年 Y の予測は、Y 年より前（Y-01-01 の 10 日前まで）で学習したモデルが出す。2022〜2025 の 4 年分。
- 単勝は予測オッズをレース内で Σ(1/オッズ) = 払戻率に比例調整（市場の構造だけを使う）。複勝は下限オッズを予測。
- 馬連・ワイド・馬単の予測オッズ = c_type / P(組)。P は予測した単勝の暗黙確率から割引 Harville（α は tp-09 で最尤推定）で作り、c_type は 2022 年の「実際のオッズ × 市場の暗黙確率」の中央値（控除と歪みの定数）。
- **買い目の候補の選び方も予測オッズだけ**（確率上位 12 点 + 予測 EV 上位 6 点 / 券種）。実際のオッズは精算にだけ使う。
- 買い目の確率はモデル単独（市場とのブレンドはしない。市場のオッズを見ないので）。

## 比べるもの
- 予測オッズで判断する方策（RL）、基準（モデル確率の 1 位に全額、予測 EV 上位 k 点）。
- 参考: 確定オッズで判断する RL（tp-09。楽観側の上限）。
- 2 段階（実運用の流れ）: 前日に予測オッズで候補を絞り、**締切 15 分前のオッズ（確定 × 雑音で代用）で期待値を確認**して外れたものを落とす。

評価: fold A = 学習 2022 → 評価 2023、fold B = 学習 2022〜2023 → 評価 2024、最終 = 学習 2022〜2024 → test 2025（一度だけ）。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, time, gc
from datetime import datetime
import numpy as np
import pandas as pd
import mlflow
import lightgbm as lgb

from src.pipeline.betting import policy as pol
from src.pipeline.betting import pred_odds as po
from src.pipeline.betting import ticket_env as te
from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
BET = REPO_ROOT / "data/local/modeling/betting"
TAG = os.environ.get("TP12_TAG", "pred_v1")
SMOKE = os.environ.get("TP12_SMOKE") == "1"
OUT = BET / "pred_odds" / TAG
OUT.mkdir(parents=True, exist_ok=True)
LOCK = BET / "pred_odds" / f"TEST2025_EVALUATED_{TAG}.json"
THREADS = int(os.environ.get("TP12_THREADS", max(1, min(14, detect_compute(reserve_cores=6).threads))))
SEEDS = (0,) if SMOKE else (0, 1, 2)
FINAL_SEEDS = (0, 1) if SMOKE else (0, 1, 2, 3, 4)
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
mlflow.set_experiment("keiba-tp-betting-predicted-odds")
require_tracking_server()
YEARS = ["2022", "2023", "2024", "2025"]
wp = pd.read_parquet(BET / "odds_win_place.parquet"); wp = wp[wp["race_id"].str[:4].isin(YEARS) & (wp["win_odds"] > 0)].reset_index(drop=True)
pay = pd.read_parquet(BET / "payoff.parquet"); pay = pay[pay["race_id"].str[:4].isin(YEARS)].reset_index(drop=True)
cols = ["race_id", "horse_id", "year", "p_win", "p_top2", "p_top3"]
pred_all = pd.concat([pd.read_parquet(WORK / "oof/model_oof/ensemble_v2.parquet")[cols], pd.read_parquet(WORK / "final_eval_v2/test2025_predictions.parquet")[cols]], ignore_index=True)
pred_all = pred_all[pred_all["year"].between(2022, 2025)]
BEST8 = json.loads((BET / "rl/ens_v2/best_config.json").read_text(encoding="utf-8"))["params"]
CMP = BET / "rl_compare/cmp_v2"
TDIR = BET / "tickets/cmp_v2_K12_v2"
TICK_ACT = {y: pd.read_parquet(TDIR / f"{y}.parquet") for y in YEARS}          # 確定オッズの買い目（参考と、2段階の確認に使う）
AL = pd.read_csv(CMP / "alpha_mle.csv", index_col=0)
AL2, AL3 = float(AL.loc["最尤推定（2022 年）", "α2"]), float(AL.loc["最尤推定（2022 年）", "α3"])
CONSTS = po.pair_constants(TICK_ACT["2022"])
print("α2, α3 =", round(AL2, 3), round(AL3, 3), "| c_type =", {k: round(v, 3) for k, v in CONSTS.items()})

## 1. オッズ予測（ローリング、オッズ関連情報なし）

In [ ]:
wp_all = pd.read_parquet(BET / "odds_win_place.parquet"); wp_all = wp_all[wp_all["win_odds"] > 0]          # 学習ターゲット（過去年の確定オッズ）は 2020 年から使う
D = tp05_data.load_dataset(WORK)
df = D["df"]
feats = D["FEATS"]["base+S+C1"]
po.assert_no_odds_features(feats)            # オッズ・人気に関わる列が混ざっていれば停止
print(f"特徴量 {len(feats)} 列（オッズ・人気系なし）")
T = df[["race_id", "horse_id", "date"]].reset_index().merge(wp_all[["race_id", "horse_id", "win_odds", "place_odds_min"]], on=["race_id", "horse_id"], how="inner")
X = df.loc[T["index"], feats]
dates = pd.to_datetime(T["date"]); yr = dates.dt.year.to_numpy()
ylog = {"win": np.log(T["win_odds"].to_numpy()), "place": np.log(np.clip(T["place_odds_min"].fillna(1.1).to_numpy(), 1.0, None))}
P = {"objective": "regression", "learning_rate": 0.05, "num_leaves": 63, "min_data_in_leaf": 50, "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 1.0,
     "verbosity": -1, "num_threads": THREADS, "seed": 0, "deterministic": True, "force_row_wise": True}
HL = None
rep = WORK / "odds_drift/report.json"
if rep.exists():
    prop = json.loads(rep.read_text(encoding="utf-8")).get("proposal")
    print("tp-10 の提案:", prop)
    HL = {"半減期1年": 1.0, "半減期2年": 2.0}.get(prop)
rows, preds = [], []
for Y in ([2023, 2024] if SMOKE else [2022, 2023, 2024, 2025]):
    t0 = time.time()
    tr = np.flatnonzero(dates < pd.Timestamp(f"{Y}-01-01") - pd.Timedelta(days=10)); te_ = np.flatnonzero(yr == Y)
    w = (0.5 ** (((Y - 1) - yr[tr]) / HL)) if HL else None
    out = T.iloc[te_][["race_id", "horse_id"]].copy()
    for k in ("win", "place"):
        b = lgb.train(P, lgb.Dataset(X.iloc[tr], label=ylog[k][tr], weight=w), num_boost_round=500)
        out[f"{k}_raw"] = np.exp(b.predict(X.iloc[te_]))
        r = ylog[k][te_] - np.log(out[f"{k}_raw"].to_numpy())
        rows.append({"検証年": Y, "対象": k, "rmse_log": float(np.sqrt((r ** 2).mean())), "偏り": float(r.mean()), "sec": time.time() - t0})
    preds.append(out); gc.collect()
    print(Y, f"{time.time() - t0:.0f}s", flush=True)
PO = pd.concat(preds, ignore_index=True)
PO["win_pred"] = po.normalize_win_odds(PO["race_id"].to_numpy(), PO["win_raw"].to_numpy())
PO["place_pred"] = np.clip(PO["place_raw"], 1.0, None)
acc = pd.DataFrame(rows); display(acc.round(4)); acc.to_csv(OUT / "odds_prediction_accuracy.csv", index=False)
chk = PO.merge(wp[["race_id", "horse_id", "win_odds", "place_odds_min"]], on=["race_id", "horse_id"])
print("正規化後の単勝: 対数 RMSE", round(float(np.sqrt(((np.log(chk.win_odds) - np.log(chk.win_pred)) ** 2).mean())), 4), "/ 中央の予測÷実際", round(float((chk.win_pred / chk.win_odds).median()), 3))

## 2. 予測オッズで判断する買い目の表
実際のオッズは使わない（馬の出走確認と払戻の精算にだけ使う）。

In [ ]:
TP = {}
TD = BET / "tickets" / f"{TAG}_K12"
TD.mkdir(parents=True, exist_ok=True)
for y in (["2023", "2024"] if SMOKE else YEARS):
    f = TD / f"{y}.parquet"; mf = TD / f"{y}.json"
    want = {"al": [round(AL2, 4), round(AL3, 4)], "consts": {k: round(v, 4) for k, v in CONSTS.items()}, "hl": HL, "ver": 1}
    if f.exists() and mf.exists() and json.loads(mf.read_text()) == want:
        TP[y] = pd.read_parquet(f); print(y, "キャッシュ", len(TP[y])); continue
    t0 = time.time()
    ids = set(pred_all.loc[pred_all["year"] == int(y), "race_id"])
    d = te.build_tickets_predicted_parallel(pred_all[pred_all["race_id"].isin(ids)], wp[wp["race_id"].isin(ids)], pay[pay["race_id"].isin(ids)], PO[PO["race_id"].isin(ids)], CONSTS, (AL2, AL3), K=12, procs=min(THREADS, 6), chunks=40)
    d.to_parquet(f, index=False); mf.write_text(json.dumps(want)); TP[y] = d
    print(f"{y}: {d['race_id'].nunique():,} / {len(ids):,} レース、{len(d):,} 点 ({time.time() - t0:.0f}s)", flush=True)

## 3. 戦略の比較
RL は tp-08 の最良設定（賭け金の下限 0.5）。確率はモデル単独、オッズは予測。

In [ ]:
def rl_alloc(tr_years, te_year, seeds):
    dtr = pd.concat([TP[y] for y in tr_years], ignore_index=True)
    dte = TP[te_year].reset_index(drop=True)
    pk_tr, mu, sd = pol.pack(dtr)
    pk_te, _, _ = pol.pack(dte, mu, sd)
    ws = []
    for s in seeds:
        c = pol.TrainConfig(**{**BEST8, "seed": s, "threads": min(THREADS, 4)})
        net, _ = pol.train_policy(pk_tr, c)
        ws.append(pol.predict_alloc(net, pk_te, c.max_ticket))
    return pk_te, dte, np.mean(ws, axis=0)


def baselines(pk, dte):
    ev = pol.pack_column(pk, dte.assign(_ev=dte["p_comb"] * dte["odds"]), "_ev")
    return {
        "モデル確率1位: 単勝": pol.alloc_top_p(pk, "tansho"), "モデル確率1位: 複勝": pol.alloc_top_p(pk, "fukusho"), "モデル確率1位: 馬連": pol.alloc_top_p(pk, "umaren"),
        "予測EV上位3点（参加100%）": pol.alloc_topk_score(pk, ev, 3, 1.0), "予測EV上位3点（参加50%）": pol.alloc_topk_score(pk, ev, 3, 0.5), "ランダム1点": pol.alloc_random(pk),
    }


def confirm(pk, dte, w, year, sigma, tau, seed=0):
    """2 段階: 締切 15 分前のオッズ（確定 × 雑音）と、市場とのブレンド確率で期待値を確認し、tau 未満の買い目を落とす。"""
    act = TICK_ACT[year][["race_id", "type", "combo", "p_comb", "odds"]].rename(columns={"p_comb": "p_blend", "odds": "odds_act"})
    m = dte[["race_id", "type", "combo"]].merge(act, on=["race_id", "type", "combo"], how="left")
    rng = np.random.default_rng(seed)
    o15 = m["odds_act"].to_numpy() * np.exp(rng.normal(0, sigma, len(m))) if sigma > 0 else m["odds_act"].to_numpy()
    ok = (m["p_blend"].to_numpy() * o15 >= tau)
    keep = pol.pack_column(pk, dte.assign(_k=np.where(np.isnan(ok.astype(float)) | ~np.isfinite(m["p_blend"].to_numpy() * o15), 0.0, ok.astype(float))), "_k")
    return w * np.nan_to_num(keep, nan=0.0)


def run_period(tr_years, te_year, seeds, n_boot=500):
    pk, dte, w = rl_alloc(tr_years, te_year, seeds)
    allocs = {"RL（予測オッズ）": w, **baselines(pk, dte)}
    for sg in (0.0, 0.1):
        for tau in (0.85, 1.0):
            allocs[f"RL → T-15確認（σ={sg}, EV≥{tau}）"] = confirm(pk, dte, w, te_year, sg, tau)
    rows = []
    for k, a in allocs.items():
        s = pol.summarize(pol.settle(pk, a), n_boot=n_boot)
        rows.append({"方策": k, **{c: s[c] for c in ("roi", "roi_lo", "roi_hi", "profit_per_race", "stake_per_race", "participation")}})
    # 選ばれた買い目の 予測オッズ ÷ 確定オッズ（賭け金で重みづけ）: 1 を超えれば、予測が実際より高く、判断が楽観的
    act = TICK_ACT[te_year][["race_id", "type", "combo", "odds"]].rename(columns={"odds": "odds_act"})
    m = dte[["race_id", "type", "combo", "odds"]].merge(act, on=["race_id", "type", "combo"], how="left")
    ratio = pol.pack_column(pk, dte.assign(_r=(m["odds"] / m["odds_act"]).to_numpy()), "_r")
    ww = np.where(pk.mask & np.isfinite(ratio), w, 0.0)
    bias = float((ww * np.nan_to_num(ratio, nan=0.0).clip(0, 10)).sum() / max(ww.sum(), 1e-12)) if np.isfinite(ratio).any() else np.nan
    return pd.DataFrame(rows), bias, pk, allocs


FOLDS = [("A", ["2022"], "2023"), ("B", ["2022", "2023"], "2024")] if not SMOKE else [("B", ["2023"], "2024")]
VAL = []
for fold, tr_y, va_y in FOLDS:
    t0 = time.time()
    r, bias, _, _ = run_period(tr_y, va_y, SEEDS)
    r.insert(0, "fold", fold); r["選ばれた買い目の予測÷確定オッズ"] = bias
    VAL.append(r); print(fold, f"{time.time() - t0:.0f}s 予測÷確定 {bias:.3f}", flush=True)
VAL = pd.concat(VAL, ignore_index=True)
display(VAL.pivot_table(index="方策", columns="fold", values="roi").round(3)); VAL.to_csv(OUT / "valid_compare.csv", index=False)

## 4. 最終: 学習 2022〜2024 → test 2025（一度だけ）
参考として、確定オッズで判断する RL（tp-09）の結果も並べる。

In [ ]:
if not SMOKE and LOCK.exists() and os.environ.get("TP12_FORCE") != "1":
    raise SystemExit(f"test 2025 は評価済みです（{LOCK}）。結果を見てやり直さないため、再実行しません。")
TEST_Y = "2024" if SMOKE else "2025"
TR_Y = ["2023"] if SMOKE else ["2022", "2023", "2024"]
FIN, bias, pk_f, allocs_f = run_period(TR_Y, TEST_Y, FINAL_SEEDS, n_boot=2000)
FIN["選ばれた買い目の予測÷確定オッズ"] = bias
ref = CMP / "final_test2025.csv"
if ref.exists():
    r0 = pd.read_csv(ref)
    FIN = pd.concat([FIN, r0[r0["方策"].isin(["RL 基本（tp-08）", "複勝: モデル1位に全額"])].assign(方策=lambda d: "（参考・確定オッズで判断）" + d["方策"])[FIN.columns.intersection(r0.columns)]], ignore_index=True)
display(FIN.round(4)); FIN.to_csv(OUT / "final_test2025.csv", index=False)
print(f"選ばれた買い目の 予測オッズ ÷ 確定オッズ（賭け金で重みづけ）= {bias:.3f}")
res = {"evaluated_at": datetime.now().isoformat(timespec="seconds"), "tag": TAG, "git_commit": GIT_COMMIT, "test_year": TEST_Y, "smoke": SMOKE, "final": FIN.to_dict("records"), "valid": VAL.to_dict("records"),
       "odds_prediction_accuracy": acc.to_dict("records"), "recency_halflife": HL, "alphas": [AL2, AL3], "consts": CONSTS,
       "notes": ["オッズ予測の特徴量にオッズ・人気は含めない（列名検査）", "精算は確定の払戻", "2 段階の T-15 は確定 × 雑音による粗い代用"]}
(OUT / "report.json").write_text(json.dumps(res, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
if not SMOKE:
    with mlflow.start_run(run_name=f"tp-12-predicted-odds-{TAG}", tags={"notebook": "tp-12", "git_commit": GIT_COMMIT, "test_year": "2025"}):
        for i_, (_, r) in enumerate(FIN.iterrows()):
            if pd.notna(r.get("roi")):
                mlflow.log_metrics({f"test2025_roi__{i_:02d}": float(r["roi"])})
        mlflow.log_dict({f"{i_:02d}": r["方策"] for i_, (_, r) in enumerate(FIN.iterrows())}, "pred_odds/metric_index.json")
        for p in OUT.glob("*"):
            mlflow.log_artifact(str(p), "pred_odds")
    LOCK.write_text(json.dumps({"evaluated_at": res["evaluated_at"], "git_commit": GIT_COMMIT, "note": "test 2025 の予測オッズ評価は実施済み。結果を見てやり直さない。"}, ensure_ascii=False))
print("完了")